# RAG 101 — Corpus + Answerability, over **Ollama**

> *Corpus:* IBM mt-rag-benchmark government-services passages (subset of the docs).

**Duration:** ~1 min if the ChromaDB index already exists; first build downloads the embedding model + corpus.

This is the [`rag_101.ipynb`](../granite-switch/tutorials/notebooks/rag_101.ipynb) tutorial, adapted to
run against a local **`ollama serve`** instead of vLLM. It's the smallest end-to-end RAG demo: build a
vector corpus, retrieve passages for a query, and ask the model **“can these passages actually answer
it?”** — no generation, no citations, just the gate that decides whether RAG should even attempt an answer.

**What changes vs. upstream:**
- Retrieval is identical — same `load_or_build_govt_chroma` loader (vendored here as `rag_corpus.py`),
  same `granite-embedding` model, same ChromaDB index. This repo's loader also prefers **Apple Metal (MPS)**
  for the embedder, so retrieval runs on the GPU too.
- The answerability adapter call goes through `backend.call_adapter("answerability", …)` instead of
  `rag.check_answerability(...)`. Same Mellea io.yaml, same envelope.

**Adapter used:** the `answerability` intrinsic from the [RAG](https://huggingface.co/ibm-granite/granitelib-rag-r1.0) library.

## Prerequisites

1. The **patched** `ollama serve` running with the `granite-switch` model created (see [`README.md`](./README.md)).
2. Run under the project venv (`chromadb`, `sentence-transformers`, `torch` are project deps).
3. First corpus build downloads ~50 MB (govt.jsonl) + the embedding model; later runs load `./govt_chroma` instantly.

No GPU server to launch, no HuggingFace login.

## 1 · Configuration

Endpoints, model IDs, and corpus paths. The backend's corpus defaults already match the tutorial
(`./govt_chroma`, `granite-embedding-small-english-r2`, tutorial-docs subset), so we mostly just set
the Ollama target and `TOP_K`.

In [1]:
import os
from ollama_intrinsic import OllamaIntrinsicBackend

OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434")
MODEL = os.environ.get("GRANITE_SWITCH_MODEL", "granite-switch")
GGUF = os.environ.get("GRANITE_SWITCH_GGUF", "granite-switch-4.1-3b-preview-f16.gguf")

# TOP_K = passages retrieved per query. Small here because we only feed them to
# answerability — no generation budget to worry about.
TOP_K = 5

print(f"Ollama:    {OLLAMA_URL}  ({MODEL})")
print(f"Embedding: ibm-granite/granite-embedding-small-english-r2")
print(f"ChromaDB:  ./govt_chroma")

Ollama:    http://127.0.0.1:11434  (granite-switch)
Embedding: ibm-granite/granite-embedding-small-english-r2
ChromaDB:  ./govt_chroma


## 2 · Build or load the vector corpus

`backend.ensure_corpus()` is the corpus half of RAG. Internally it calls the vendored
`load_or_build_govt_chroma` (in `rag_corpus.py`):

1. Downloads `govt.jsonl.zip` (subset of 49k government-service passages from IBM mt-rag-benchmark) on first run.
2. Embeds each passage with `granite-embedding-small-english-r2` (on MPS/Metal if available).
3. Persists the index to `./govt_chroma` so subsequent runs load instantly.

> To embed the full corpus instead of the curated subset, construct the backend with
> `OllamaIntrinsicBackend(..., load_only_tutorial_docs=False)`.

In [2]:
backend = OllamaIntrinsicBackend(model=MODEL, ollama_url=OLLAMA_URL, gguf_path=GGUF)
backend.ensure_corpus()
print(f"Corpus ready — {backend._collection.count():,} passages indexed.")

/Users/barha/Desktop/IBM/projects/granite-switch-ollama-mellea/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 74/74 [00:00<00:00, 10035.19it/s]

Granite embedding model ready on mps  (ibm-granite/granite-embedding-small-english-r2)
Loaded from ./govt_chroma  (137 docs).
Corpus ready — 137 passages indexed.


## 3 · Retrieve passages for an answerable query

`backend.retrieve(query, top_k=TOP_K)` runs the dense (cosine) search over the corpus and returns the
top-K passage texts — the same `chroma_collection.query(...)` the upstream notebook runs.

In [3]:
query = "How long does it take for the IRS to issue a tax refund?"
passages = backend.retrieve(query, top_k=TOP_K)

print(f"query: {query}\n")
for i, p in enumerate(passages):
    print(f"[{i}] {p[:160]}{'…' if len(p) > 160 else ''}\n")

query: How long does it take for the IRS to issue a tax refund?

[0] What to Expect for Refunds This Year | Internal Revenue Service
You will get personalized refund information based on the processing of your tax return. The too…

[1] What to Expect for Refunds This Year | Internal Revenue Service
No more than three electronic refunds can be deposited into a single financial account or pre-pa…

[2] Timeframes | Wait times | California Franchise Tax Board
Mail or fax
Note
105
days
Typical wait times
Mon
Tue
Wed
Thur
Fri

Processing times 

tax return icon

…

[3] What to Expect for Refunds This Year | Internal Revenue Service
Refunds

Credits & Deductions

Forms & Instructions

Info Menu Mobile

Help

News

Charities & N…

[4] Time You Can Claim a Credit or Refund | Internal Revenue Service
File because of a bad debt deduction or a worthless security loss: You have 7 years from the re…



## 4 · Answerability — can these passages answer the query?

The answerability adapter judges the retrieved passages, returning `answerable` or `unanswerable`.
Upstream calls `rag.check_answerability(query, docs, ctx, backend)`; the bridge wraps the passages as
`{"doc_id", "text"}` dicts and calls `call_adapter("answerability", …, documents=…)`, reading
`parsed["answerability"]`.

In [4]:
def check_answerability(query, passages):
    docs = [{"doc_id": str(i), "text": t} for i, t in enumerate(passages)]
    out = backend.call_adapter(
        "answerability",
        [{"role": "user", "content": query}],
        documents=docs, num_predict=8,
    )
    return (out.get("parsed") or {}).get("answerability")

verdict = check_answerability(query, passages)
badge = "✅ answerable" if verdict == "answerable" else "🔍 unanswerable"
print(f"{badge}  (verdict={verdict!r})")

✅ answerable  (verdict='answerable')


## 5 · The **unanswerable** exit

The point of answerability is to let an application **refuse instead of hallucinate**. Here's a query
whose answer isn't in the government-services corpus — retrieval still returns its nearest neighbors,
but answerability should flag them as insufficient.

In [5]:
off_topic = "What is the airspeed velocity of an unladen swallow?"
passages2 = backend.retrieve(off_topic, top_k=TOP_K)
verdict2 = check_answerability(off_topic, passages2)

print(f"query  : {off_topic}")
print(f"top hit: {passages2[0][:120]}…\n")
badge = "✅ answerable" if verdict2 == "answerable" else "🔍 unanswerable"
print(f"{badge}  (verdict={verdict2!r})")
if verdict2 == "unanswerable":
    print("  → application should refuse: 'I don't have enough information to answer that.'")

query  : What is the airspeed velocity of an unladen swallow?
top hit: Contact us | FTB.ca.gov
Contact us | FTB.ca.gov

Skip to Main Content

News
Filing and payment due dates for San Diego C…

🔍 unanswerable  (verdict='unanswerable')
  → application should refuse: 'I don't have enough information to answer that.'


## 6 · Recap & next

Retrieval + answerability is the foundation the larger flows build on: retrieve candidates, then gate
on whether they can actually support an answer before spending any generation budget. The retrieval
half is plain ChromaDB; the gate is one Granite Switch adapter call through the Ollama bridge.

- **`rag_flow_ollama.ipynb`** — adds rewrite, clarification, grounded answering, and citations around
  this gate, as a stateful 7-turn conversation.
- **`hello_mellea_ollama.ipynb`** — each adapter function in isolation on hardcoded docs.